In [1]:
import json
import os

In [2]:
import numpy as np
import pandas as pd

In [27]:
SPLITS_DIR = "preprocessing/splits"
ARTIFACTS_DIR = "preprocessing/artifacts"
OUTPUT_DIR = "preprocessing/output"

In [28]:
 
TRAIN_CSV = os.path.join(SPLITS_DIR, "train.csv")
VAL_CSV = os.path.join(SPLITS_DIR, "val.csv")
TEST_CSV = os.path.join(SPLITS_DIR, "test.csv")


## Task A

In [29]:
SCALER = os.path.join(ARTIFACTS_DIR, "scaler.joblib")
ENCODER = os.path.join(ARTIFACTS_DIR, "onehot_encoder.joblib")
CLASS_MAP = os.path.join(ARTIFACTS_DIR, "class_map.json")
CLASS_WEIGHTS = os.path.join(ARTIFACTS_DIR, "class_weights.json")

In [30]:
X_TAB_TRAIN = os.path.join(OUTPUT_DIR, "X_tab_train.npy")
X_TAB_VAL = os.path.join(OUTPUT_DIR, "X_tab_val.npy")
X_TAB_TEST = os.path.join(OUTPUT_DIR, "X_tab_test.npy")
Y_TRAIN = os.path.join(OUTPUT_DIR, "y_train.npy")
Y_VAL = os.path.join(OUTPUT_DIR, "y_val.npy")
Y_TEST = os.path.join(OUTPUT_DIR, "y_test.npy")

## Task B

In [31]:
VOCAB_SPEC = os.path.join(ARTIFACTS_DIR, "vocab.txt")
VOCAB_REAL = "preprocessing/vocab.txt" 

In [32]:
X_TEXT_TRAIN_SPEC = os.path.join(OUTPUT_DIR, "X_text_train.npy")
X_TEXT_VAL_SPEC = os.path.join(OUTPUT_DIR, "X_text_val.npy")
X_TEXT_TEST_SPEC = os.path.join(OUTPUT_DIR, "X_text_test.npy")

In [33]:
X_TEXT_TRAIN_REAL = "preprocessing/X_text_train.npy"
X_TEXT_VAL_REAL = "preprocessing/X_text_val.npy"
X_TEXT_TEST_REAL = "preprocessing/X_text_test.npy"

In [34]:
PASS = "[PASS]"
FAIL = "[FAIL]"
WARN = "[WARN]"

In [35]:
results = []


In [36]:
def check(test_id, description, passed, detail=""):
    status = PASS if passed else FAIL
    results.append({"id": test_id, "description": description, "status": status, "detail": detail})
    print(f"  {status} {test_id}: {description}")
    if detail:
        print(f"         {detail}")
    return passed
 

In [37]:
def warn(test_id, description, detail=""):
    results.append({"id": test_id, "description": description, "status": WARN, "detail": detail})
    print(f"  {WARN} {test_id}: {description}")
    if detail:
        print(f"         {detail}")
 

In [38]:
def file_exists(path):
    return os.path.isfile(path)

In [39]:
def resolve_path(spec_path, real_path):
    """Devuelve la ruta que existe, priorizando la SPEC."""
    if file_exists(spec_path):
        return spec_path, True  # (path, en_ruta_correcta)
    if file_exists(real_path):
        return real_path, False
    return None, False
 

# AUDITORIA TAREA A — PREPROCESAMIENTO TABULAR


In [40]:
print("=" * 65)
print("AUDITORIA TAREA A — PREPROCESAMIENTO TABULAR")
print("=" * 65)

AUDITORIA TAREA A — PREPROCESAMIENTO TABULAR


In [41]:

print("\n--- A1: Existencia de archivos ---")
a_files = {
    "train.csv": TRAIN_CSV,
    "val.csv": VAL_CSV,
    "test.csv": TEST_CSV,
    "scaler.joblib": SCALER,
    "onehot_encoder.joblib": ENCODER,
    "class_map.json": CLASS_MAP,
    "class_weights.json": CLASS_WEIGHTS,
    "X_tab_train.npy": X_TAB_TRAIN,
    "X_tab_val.npy": X_TAB_VAL,
    "X_tab_test.npy": X_TAB_TEST,
    "y_train.npy": Y_TRAIN,
    "y_val.npy": Y_VAL,
    "y_test.npy": Y_TEST,
}


--- A1: Existencia de archivos ---


In [42]:
for name, path in a_files.items():
    check(f"A1-{name}", f"Archivo existe: {path}", file_exists(path))
 

  [PASS] A1-train.csv: Archivo existe: preprocessing/splits/train.csv
  [PASS] A1-val.csv: Archivo existe: preprocessing/splits/val.csv
  [PASS] A1-test.csv: Archivo existe: preprocessing/splits/test.csv
  [PASS] A1-scaler.joblib: Archivo existe: preprocessing/artifacts/scaler.joblib
  [PASS] A1-onehot_encoder.joblib: Archivo existe: preprocessing/artifacts/onehot_encoder.joblib
  [PASS] A1-class_map.json: Archivo existe: preprocessing/artifacts/class_map.json
  [PASS] A1-class_weights.json: Archivo existe: preprocessing/artifacts/class_weights.json
  [PASS] A1-X_tab_train.npy: Archivo existe: preprocessing/output/X_tab_train.npy
  [PASS] A1-X_tab_val.npy: Archivo existe: preprocessing/output/X_tab_val.npy
  [PASS] A1-X_tab_test.npy: Archivo existe: preprocessing/output/X_tab_test.npy
  [PASS] A1-y_train.npy: Archivo existe: preprocessing/output/y_train.npy
  [PASS] A1-y_val.npy: Archivo existe: preprocessing/output/y_val.npy
  [PASS] A1-y_test.npy: Archivo existe: preprocessing/output

In [43]:
print("\n--- A2: Integridad de splits ---")
if all(file_exists(p) for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]):
    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VAL_CSV)
    test_df = pd.read_csv(TEST_CSV)
 
    total = len(train_df) + len(val_df) + len(test_df)
    check("A2-total", f"Splits suman 3000 filas", total == 3000,
          f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}, total={total}")
 
    ids_train = set(train_df["id_diagnostico"])
    ids_val = set(val_df["id_diagnostico"])
    ids_test = set(test_df["id_diagnostico"])
    overlap_tv = ids_train & ids_val
    overlap_tt = ids_train & ids_test
    overlap_vt = ids_val & ids_test
    no_overlap = len(overlap_tv) == 0 and len(overlap_tt) == 0 and len(overlap_vt) == 0
    check("A2-no-overlap", "Sin IDs compartidos entre splits", no_overlap,
          f"train&val={len(overlap_tv)}, train&test={len(overlap_tt)}, val&test={len(overlap_vt)}")
 
    # respuesta_texto presente (necesario para Tarea B)
    check("A2-texto-col", "Columna respuesta_texto presente en splits",
          "respuesta_texto" in train_df.columns)
 
    # --- A3: Estratificacion ---
    print("\n--- A3: Estratificacion del target ---")
    for name, sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
        dist = (sdf["nivel_madurez"].value_counts(normalize=True) * 100).round(1)
        print(f"    {name}: {dist.to_dict()}")
 
    # Verificar que la diferencia maxima entre splits es < 2pp por clase
    dists = {}
    for name, sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
        dists[name] = sdf["nivel_madurez"].value_counts(normalize=True) * 100
    all_classes = dists["train"].index
    max_diff = 0
    for cls in all_classes:
        vals = [dists[s].get(cls, 0) for s in ["train", "val", "test"]]
        diff = max(vals) - min(vals)
        max_diff = max(max_diff, diff)
    check("A3-stratified", f"Diferencia maxima entre splits < 2pp",
          max_diff < 2.0, f"max_diff={max_diff:.2f}pp")
 


--- A2: Integridad de splits ---
  [PASS] A2-total: Splits suman 3000 filas
         train=2250, val=450, test=300, total=3000
  [PASS] A2-no-overlap: Sin IDs compartidos entre splits
         train&val=0, train&test=0, val&test=0
  [PASS] A2-texto-col: Columna respuesta_texto presente en splits

--- A3: Estratificacion del target ---
    train: {'En Desarrollo': 30.8, 'Definido': 30.4, 'Inicial': 26.1, 'Optimizado': 12.8}
    val: {'En Desarrollo': 30.7, 'Definido': 30.4, 'Inicial': 26.0, 'Optimizado': 12.9}
    test: {'En Desarrollo': 30.7, 'Definido': 30.3, 'Inicial': 26.3, 'Optimizado': 12.7}
  [PASS] A3-stratified: Diferencia maxima entre splits < 2pp
         max_diff=0.33pp


In [44]:

# --- A4: Shapes de tensores tabulares ---
print("\n--- A4: Shapes de tensores ---")
expected_shapes = {
    "X_tab_train": (2250, 10), "X_tab_val": (450, 10), "X_tab_test": (300, 10),
    "y_train": (2250,), "y_val": (450,), "y_test": (300,),
}
npy_paths = {
    "X_tab_train": X_TAB_TRAIN, "X_tab_val": X_TAB_VAL, "X_tab_test": X_TAB_TEST,
    "y_train": Y_TRAIN, "y_val": Y_VAL, "y_test": Y_TEST,
}
loaded_npy = {}
for name, path in npy_paths.items():
    if file_exists(path):
        arr = np.load(path)
        loaded_npy[name] = arr
        expected = expected_shapes[name]
        check(f"A4-{name}", f"Shape {name}: esperado {expected}",
              arr.shape == expected, f"real={arr.shape}")
    else:
        check(f"A4-{name}", f"Shape {name}", False, "archivo no encontrado")
 


--- A4: Shapes de tensores ---
  [PASS] A4-X_tab_train: Shape X_tab_train: esperado (2250, 10)
         real=(2250, 10)
  [PASS] A4-X_tab_val: Shape X_tab_val: esperado (450, 10)
         real=(450, 10)
  [PASS] A4-X_tab_test: Shape X_tab_test: esperado (300, 10)
         real=(300, 10)
  [PASS] A4-y_train: Shape y_train: esperado (2250,)
         real=(2250,)
  [PASS] A4-y_val: Shape y_val: esperado (450,)
         real=(450,)
  [PASS] A4-y_test: Shape y_test: esperado (300,)
         real=(300,)


In [45]:

# --- A5: Escalado correcto (medias ~0, std ~1 en train) ---
print("\n--- A5: Evidencia de escalado en train ---")
if "X_tab_train" in loaded_npy:
    X = loaded_npy["X_tab_train"]
    num_cols = X[:, :2]  # las 2 primeras columnas son las numericas escaladas
    means = num_cols.mean(axis=0)
    stds = num_cols.std(axis=0)
    means_ok = all(abs(m) < 0.05 for m in means)
    stds_ok = all(abs(s - 1.0) < 0.1 for s in stds)
    check("A5-mean", "Medias de numericas en train cercanas a 0",
          means_ok, f"means={np.round(means, 4)}")
    check("A5-std", "Desviaciones de numericas en train cercanas a 1",
          stds_ok, f"stds={np.round(stds, 4)}")
 


--- A5: Evidencia de escalado en train ---
  [PASS] A5-mean: Medias de numericas en train cercanas a 0
         means=[-0.  0.]
  [PASS] A5-std: Desviaciones de numericas en train cercanas a 1
         stds=[1. 1.]


In [46]:

# --- A6: class_map.json correcto ---
print("\n--- A6: Mapeo de clases ---")
if file_exists(CLASS_MAP):
    with open(CLASS_MAP) as f:
        cmap = json.load(f)
    expected_map = {"Inicial": 0, "En Desarrollo": 1, "Definido": 2, "Optimizado": 3}
    check("A6-classmap", "class_map.json coincide con SPEC",
          cmap == expected_map, f"real={cmap}")
else:
    check("A6-classmap", "class_map.json existe", False)
 
# --- A7: class_weights.json ---
print("\n--- A7: Class weights ---")
if file_exists(CLASS_WEIGHTS):
    with open(CLASS_WEIGHTS) as f:
        cw = json.load(f)
    check("A7-keys", "class_weights tiene 4 clases",
          len(cw) == 4, f"keys={list(cw.keys())}")
    # Optimizado (clase 3) debe tener el peso mas alto
    if len(cw) == 4:
        w3 = float(cw.get("3", cw.get(3, 0)))
        max_key = max(cw, key=lambda k: float(cw[k]))
        check("A7-optimizado", "Clase Optimizado (3) tiene el peso mas alto",
              str(max_key) == "3", f"peso_clase_3={w3:.3f}, max_key={max_key}")
else:
    check("A7-keys", "class_weights.json existe", False)
 
# --- A8: Valores del target ---
print("\n--- A8: Rango del target ---")
if "y_train" in loaded_npy:
    y_all = np.concatenate([loaded_npy[k] for k in ["y_train", "y_val", "y_test"]])
    unique_vals = sorted(np.unique(y_all))
    check("A8-range", "Target contiene exactamente {0,1,2,3}",
          unique_vals == [0, 1, 2, 3], f"valores unicos={unique_vals}")
 


--- A6: Mapeo de clases ---
  [PASS] A6-classmap: class_map.json coincide con SPEC
         real={'Inicial': 0, 'En Desarrollo': 1, 'Definido': 2, 'Optimizado': 3}

--- A7: Class weights ---
  [PASS] A7-keys: class_weights tiene 4 clases
         keys=['0', '1', '2', '3']
  [PASS] A7-optimizado: Clase Optimizado (3) tiene el peso mas alto
         peso_clase_3=1.960, max_key=3

--- A8: Rango del target ---
  [PASS] A8-range: Target contiene exactamente {0,1,2,3}
         valores unicos=[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


# AUDITORIA TAREA B — PREPROCESAMIENTO DE TEXTO

In [47]:

print("\n" + "=" * 65)
print("AUDITORIA TAREA B — PREPROCESAMIENTO DE TEXTO")
print("=" * 65)
 
# --- B1: Existencia de archivos (SPEC vs real) ---
print("\n--- B1: Existencia y ubicacion de archivos ---")
 
vocab_path, vocab_correct_loc = resolve_path(VOCAB_SPEC, VOCAB_REAL)
check("B1-vocab-exists", "vocab.txt existe", vocab_path is not None)
check("B1-vocab-location", f"vocab.txt en ruta SPEC ({VOCAB_SPEC})",
      vocab_correct_loc,
      f"encontrado en: {vocab_path}" if vocab_path else "no encontrado")
 
text_files = {
    "X_text_train.npy": (X_TEXT_TRAIN_SPEC, X_TEXT_TRAIN_REAL),
    "X_text_val.npy": (X_TEXT_VAL_SPEC, X_TEXT_VAL_REAL),
    "X_text_test.npy": (X_TEXT_TEST_SPEC, X_TEXT_TEST_REAL),
}
text_npy_paths = {}
for name, (spec_p, real_p) in text_files.items():
    path, correct = resolve_path(spec_p, real_p)
    text_npy_paths[name] = path
    check(f"B1-{name}-exists", f"{name} existe", path is not None)
    check(f"B1-{name}-location", f"{name} en ruta SPEC ({spec_p})",
          correct,
          f"encontrado en: {path}" if path else "no encontrado")
 
# --- B2: Shapes de tensores de texto ---
print("\n--- B2: Shapes de tensores de texto ---")
expected_text_shapes = {
    "X_text_train.npy": (2250, 16),
    "X_text_val.npy": (450, 16),
    "X_text_test.npy": (300, 16),
}
loaded_text = {}
for name, expected_shape in expected_text_shapes.items():
    path = text_npy_paths.get(name)
    if path:
        arr = np.load(path)
        loaded_text[name] = arr
        check(f"B2-{name}", f"Shape {name}: esperado {expected_shape}",
              arr.shape == expected_shape, f"real={arr.shape}")
        check(f"B2-{name}-dtype", f"Dtype {name} es entero",
              np.issubdtype(arr.dtype, np.integer), f"dtype={arr.dtype}")
    else:
        check(f"B2-{name}", f"Shape {name}", False, "archivo no encontrado")
 
# --- B3: Validacion del vocabulario ---
print("\n--- B3: Validacion de vocab.txt ---")
if vocab_path:
    with open(vocab_path, encoding="utf-8") as f:
        vocab_lines = [line.rstrip("\n") for line in f]
 
    check("B3-size", f"Vocabulario tiene entre 50 y 200 tokens",
          50 <= len(vocab_lines) <= 200, f"tamano={len(vocab_lines)}")
 
    if len(vocab_lines) >= 2:
        check("B3-padding", "Posicion 0 es string vacio (padding)",
              vocab_lines[0] == "", f"pos_0='{vocab_lines[0]}'")
        check("B3-unk", "Posicion 1 es [UNK]",
              vocab_lines[1] == "[UNK]", f"pos_1='{vocab_lines[1]}'")
 
    # Buscar tokens con tilde (evidencia de que no las elimino)
    accented = [t for t in vocab_lines if any(c in t for c in "áéíóúñ")]
    check("B3-tildes", "Vocabulario conserva tildes",
          len(accented) > 0, f"tokens con tilde: {accented[:10]}")
else:
    check("B3-size", "vocab.txt validacion", False, "archivo no encontrado")
 
# --- B4: Coherencia entre tensores de texto y vocab ---
print("\n--- B4: Coherencia tensor-vocabulario ---")
if "X_text_train.npy" in loaded_text and vocab_path:
    X_text = loaded_text["X_text_train.npy"]
    max_idx = int(X_text.max())
    vocab_size = len(vocab_lines)
    check("B4-max-idx", "Indice maximo en tensor < tamano del vocabulario",
          max_idx < vocab_size,
          f"max_idx={max_idx}, vocab_size={vocab_size}")
 
    # Verificar padding (ceros al final de secuencias cortas)
    rows_with_zeros = (X_text == 0).any(axis=1).sum()
    pct_padded = 100 * rows_with_zeros / len(X_text)
    check("B4-padding", "Secuencias muestran padding (ceros)",
          rows_with_zeros > 0, f"{rows_with_zeros}/{len(X_text)} filas ({pct_padded:.0f}%) tienen padding")
 
# --- B5: Alineacion con splits de Tarea A ---
print("\n--- B5: Alineacion texto-tabular ---")
if "X_text_train.npy" in loaded_text and "X_tab_train" in loaded_npy:
    for suffix in ["train", "val", "test"]:
        tab_name = f"X_tab_{suffix}"
        txt_name = f"X_text_{suffix}.npy"
        if tab_name in loaded_npy and txt_name in loaded_text:
            n_tab = loaded_npy[tab_name].shape[0]
            n_txt = loaded_text[txt_name].shape[0]
            check(f"B5-align-{suffix}",
                  f"Filas {suffix}: tabular ({n_tab}) == texto ({n_txt})",
                  n_tab == n_txt)
 
# --- B6: Muestra visual de vectorizacion ---
print("\n--- B6: Muestra de vectorizacion ---")
if "X_text_train.npy" in loaded_text and file_exists(TRAIN_CSV):
    train_texts = pd.read_csv(TRAIN_CSV)["respuesta_texto"].values
    X_text = loaded_text["X_text_train.npy"]
    print("    Primeros 3 ejemplos:")
    for i in range(min(3, len(train_texts))):
        print(f"    [{i}] texto: \"{train_texts[i][:80]}...\"")
        print(f"        vector: {X_text[i]}")
        print()
 


AUDITORIA TAREA B — PREPROCESAMIENTO DE TEXTO

--- B1: Existencia y ubicacion de archivos ---
  [PASS] B1-vocab-exists: vocab.txt existe
  [FAIL] B1-vocab-location: vocab.txt en ruta SPEC (preprocessing/artifacts/vocab.txt)
         encontrado en: preprocessing/vocab.txt
  [PASS] B1-X_text_train.npy-exists: X_text_train.npy existe
  [FAIL] B1-X_text_train.npy-location: X_text_train.npy en ruta SPEC (preprocessing/output/X_text_train.npy)
         encontrado en: preprocessing/X_text_train.npy
  [PASS] B1-X_text_val.npy-exists: X_text_val.npy existe
  [FAIL] B1-X_text_val.npy-location: X_text_val.npy en ruta SPEC (preprocessing/output/X_text_val.npy)
         encontrado en: preprocessing/X_text_val.npy
  [PASS] B1-X_text_test.npy-exists: X_text_test.npy existe
  [FAIL] B1-X_text_test.npy-location: X_text_test.npy en ruta SPEC (preprocessing/output/X_text_test.npy)
         encontrado en: preprocessing/X_text_test.npy

--- B2: Shapes de tensores de texto ---
  [PASS] B2-X_text_train.npy:

# RESUMEN

In [48]:
print("\n" + "=" * 65)
print("RESUMEN DE AUDITORIA")
print("=" * 65)
 
n_pass = sum(1 for r in results if r["status"] == PASS)
n_fail = sum(1 for r in results if r["status"] == FAIL)
n_warn = sum(1 for r in results if r["status"] == WARN)
n_total = len(results)
 
print(f"\n  Total de checks: {n_total}")
print(f"  {PASS}: {n_pass}")
print(f"  {FAIL}: {n_fail}")
print(f"  {WARN}: {n_warn}")
 
if n_fail > 0:
    print(f"\n  CHECKS FALLIDOS:")
    for r in results:
        if r["status"] == FAIL:
            print(f"    - {r['id']}: {r['description']}")
            if r["detail"]:
                print(f"      {r['detail']}")
 
# Resumen de archivos mal ubicados (Tarea B)
misplaced = []
if vocab_path and not file_exists(VOCAB_SPEC):
    misplaced.append(("vocab.txt", VOCAB_REAL, VOCAB_SPEC))
for name, (spec_p, real_p) in text_files.items():
    if file_exists(real_p) and not file_exists(spec_p):
        misplaced.append((name, real_p, spec_p))
 
if misplaced:
    print(f"\n  ARCHIVOS MAL UBICADOS (Tarea B):")
    print(f"  Para corregir, mover con estos comandos:\n")
    for fname, src, dst in misplaced:
        print(f"    mv {src} {dst}")
 
print("\n" + "=" * 65)


RESUMEN DE AUDITORIA

  Total de checks: 52
  [PASS]: 48
  [FAIL]: 4
  [WARN]: 0

  CHECKS FALLIDOS:
    - B1-vocab-location: vocab.txt en ruta SPEC (preprocessing/artifacts/vocab.txt)
      encontrado en: preprocessing/vocab.txt
    - B1-X_text_train.npy-location: X_text_train.npy en ruta SPEC (preprocessing/output/X_text_train.npy)
      encontrado en: preprocessing/X_text_train.npy
    - B1-X_text_val.npy-location: X_text_val.npy en ruta SPEC (preprocessing/output/X_text_val.npy)
      encontrado en: preprocessing/X_text_val.npy
    - B1-X_text_test.npy-location: X_text_test.npy en ruta SPEC (preprocessing/output/X_text_test.npy)
      encontrado en: preprocessing/X_text_test.npy

  ARCHIVOS MAL UBICADOS (Tarea B):
  Para corregir, mover con estos comandos:

    mv preprocessing/vocab.txt preprocessing/artifacts/vocab.txt
    mv preprocessing/X_text_train.npy preprocessing/output/X_text_train.npy
    mv preprocessing/X_text_val.npy preprocessing/output/X_text_val.npy
    mv preproc